# CARBAMYLATION ANALYSIS

**Authors**: Matteo Degiacomi and Hao Man

This Jupyter notebook enables selecting and curating protein atomic structures for the analysis of their lysines. The aim, is to extract structural information on each lysine, to help determining the subset of lysines that may be carbamylated. The pipeline is divided in the following steps.

* 1. [Data Gathering](#gather)
  - 1.1. [Mine UNIPROT database](#uniprot)
  - 1.2. [Mine PDB and AlphaFold databases](#pdb)
* 2. [Feature Extraction](#measure)
* 3. [Data Analysis](#analysis)
  - 3.1. [Loading and Cleaning the Data](#analysis_load)
  - 3.2. [Data Aggregation and Plotting](#analysis_aggregate)
  - 3.3. [GO Terms Extraction](#analysis_go)
* 4. [Interactive Data Visualization](#plot)

Further explanations about each of these steps are provided in the individual sections below.

Let's get started by executing the cell below (select it and press the <kbd>&#x21E7; Shift</kbd> + <kbd>&#x23ce; Enter</kbd>keys) to import all the Python packages required.

In [ ]:
import os

#data handling
import numpy as np
import pandas as pd
pd.set_option("display.max_rows", None, "display.max_columns", None)

#plotting
import plotly.express as px
import plotly.graph_objects as go
import nglview as nv
from ipywidgets import HBox, VBox
from ipywidgets import widgets

# atomic structures manipulation
import biobox as bb

#carbamylation analysis procedures
from uniprot import Uniprot
from protein import PDB
from measure import Measure
from analysis import Analysis
from viewer import Viewer

Before getting started, we should also define an output directory. If none defined, the default name will be "result".

In [ ]:
outdir = "Demo"

## 1. Data Gathering  <a class="anchor" id="gather"></a>

### 1.1. Mine UNIPROT database  <a class="anchor" id="uniprot"></a>

Load data from [UNIPROT database](https://www.uniprot.org/). The commands `from_csv_file` and `get_protein_data` can be launched multiple times, to compile a database saved in a pandas DataFrame (`Uniprot.df`). For the demo, we only use few uniprot codes from the main dataset.

We will start by loading a list of UNIPROT codes contained in a text file. Let's start with an optional step: visualizing the content of the file itself before loading it into our pipeline.

In [ ]:
df = pd.read_csv(f'input{os.sep}cyano_reviewed_first10.csv')
df['Uniprot_Entry'] #show data

Now that we know what the file contains, let's load it!

In [ ]:
UP = Uniprot()
UP.from_csv_file(f'input{os.sep}cyano_reviewed_first10.csv')
UP.df #show data

Besides loading UNIPROT codes from a text file, we can also append hand picked proteins in the database, by providing their UNIPROT code.

In [ ]:
UP.get_protein_data('Q03513')
UP.df #show data

### 1.2. Mine PDB and AlphaFold databases  <a class="anchor" id="pdb"></a>

Download and curate all proteins described in a Uniprot DataFrame (`Uniprot.df`). Files will be downloaded from the [Protein Data Bank (PDB)](https://www.rcsb.org/) and [AlphaFold database](https://alphafold.ebi.ac.uk/).Two optional parameters can be passed to the PDB constructor, `outdir` (output directory, "result" by default) and the largest acceptable gap in sequence (`gap`, 10 by default). Proteins having a consecutive number of missing aminoacids greater than this cutoff value will be disregarded.

For each entry in the database:
* raw files are downloaded from the PDB in a subfolder within the output directory (defined by `PDB.raw_dir`, "conformations" by default)
* each raw file is cleaned: heteroatoms, hydrogens and non-protein, non-standard aminoacids are amended. Ions are preserved, as they may have important effect on the feature extraction phase
* different conformations (usually from NMR ensembles) are saved in different files
* if multiple aminoacid rotamers exist, multiple files are produced
* all produced files are patched, i.e. missing parts in the sequence are added using Modeller
* all files correctly patched are saved in a different subfolder in the output directory (defined by `PDB.curated_dir`, "curated" by default).

This process will generate a new DataFrame, `PDB.df`, containing information on all the structures that have been correctly curated.

<div class="alert alert-warning">
<b>Warning:</b> depending on how many proteins have to be downloaded and curated, this operation may take a long time. AlphaFold and PDB files not needing curation are processed in few seconds. Large multimers requiring curation can take up to 1 min. Multiple conformations in a single protein structure are individually curated.
</div>

In [ ]:
pdb = PDB(outdir=outdir, gap=10)
pdb.gather_proteins(UP.df)

Show the DataFrame reporting on structures successfully downloaded and curated, and save it into a csv file. If no filename is given, file `proteins.csv` will be saved in the output directory.

In [ ]:
pdb.save_state()
pdb.df #show data

A csv file containing protein information can be loaded with the method `PDB.load_state(filename)`.

In [ ]:
pdb = PDB(outdir=outdir, gap=10)
pdb.load_state(f'{outdir}{os.sep}proteins.csv')
pdb.df #show data

## 2. Feature Extraction  <a class="anchor" id="measure"></a>

For all structures correctly curated in the previous step, calculate pKa and Solvent Accessible Surface Area (SASA) and save final result into a file. If no filename is given, the file will be called `measures.csv`, and saved into the folder defined by the parameter `outdir` ("result" by default). When creating an instance of the Measure class, a log file will also be created in `outdir`. The filename is defined by the parameter `log_path`, and is "measure_log.txt" by default.

<div class="alert alert-warning">
<b>Warning:</b> calculations typically take 2-20 sec. per curated protein structure (depending on its size and number of lysines).
</div>

In [ ]:
M = Measure(pdb.df, outdir=outdir, activate_log=True, log_path="measure_log.txt")
M.measure_dataframe()
M.save_state()

<div class="alert alert-info">
<b>Note</b>: the <code>Measure</code> object features methods for calculations and data recovery:
    
* <code>M.restart_measure()</code> restarts the operations of <code>M.measure_dataframe()</code>, if these interrupted by any reason.
* <code>M.recover_from_log(filename)</code> parses a logfile and returns a DataFrame.</div>


The data acquired during calculations is located in the `M.df` DataFrame. We can sort the data according to any key of interest, for instance UNIPROT entry followed by resid.

In [ ]:
M.df.sort_values(['Uniprot_Entry', 'Resid'])

## 3. Data Analysis  <a class="anchor" id="analysis"></a>

### 3.1. Loading and cleaning the data   <a class="anchor" id="analysis_load"></a>

The Analysis class is responsible for interpreting lysine measurement data. Creating `Analysis` requires a DataFrame containing data gathered by a `Measure` instance. This can come either from a previously saved logfile (as we shall do in the next box), or from a DataFrame in a `Measure` instance (`Measure.df`). Below, we will load data from csv file file we previously saved.

In [ ]:
df = pd.read_csv(f'Demo{os.sep}measures.csv')

#To see a more complete dataset, uncomment the following line to load a much larger precalculated dataset 
#df = pd.read_csv(f'all_cyano{os.sep}training_data_all.csv')

analysis = Analysis(df)

The loaded measurement data is now internally stored in the `Analysis.df` DataFrame.

Some lysine measurements may be suspiciously high (e.g., pKa>14) or low (e.g., pKa<1), which could be an indication that in their specific circumstance, calculations were faulty. These can be identified as follows, where the output is a new DataFrame.

In [ ]:
df_suspicious = analysis.get_extreme_values(feature='pKa', lower=1, upper=14)

Having identified suspicious measurements, we can delete them from our `Analysis.df` dataset as follows.

In [ ]:
analysis.remove_df(df_suspicious)

### 3.2. Data Aggregation and Slicing   <a class="anchor" id="analysis_aggregate"></a>

The data we have now cleaned can be aggregated. Aggregation involves calculating min, max, mean and standard deviation for each feature (a single lysine may features multiple measurements in the input dataframe). This operation is carried out by calling `Analysis.df_aggregate()`, and the data is internally stored in the DataFrame `Analysis.df_aggregated`.

In [ ]:
analysis.aggregate()
analysis.df_aggregated #show data

An alternative method to aggregate data is, for each lysine, to only retain one representative measurement of pKa and SASA. The selection criterion can either be `average` or `south_east`(the point with relatively lower pKa and higher SASA, the trade-off of the two features can be controlled by parameter `weight`, the default value is 0.5). The result of this aggregation will be stored in the DataFrame `Analysis.df_sub`.

In [ ]:
analysis.subset(analysis.df, method = 'average')
analysis.df_sub #show data

An interesting question involves comparing AlphaFold data to PDB data. The following method enables selecting only the data from structures predicted by AlphaFold. Note that any DataFrame in `Analysis` can always be interrogated with pandas syntax, e.g. via the [query](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.query.html) method.

In [ ]:
df_af = analysis.get_data_alphafold()
df_af #show data

`Analysis` features a method to produce basic plots of the data (histogram and boxplot). Note that if no UNIPROT code and/or resid id is given, the plot will use the data of the whole DataFrame.

In [ ]:
analysis.plot_graph(plot_type = 'histogram', 
                    feature = 'pKa', 
                    uniprot_entry = False, # can also specify a uniprot code and a resid ID
                    resid = False)

### 3.3. GO Terms Extraction  <a class="anchor" id="analysis_go"></a>

Every protein is be associated to a set of [Gene Ontology (GO) Terms](http://geneontology.org/). `Analysis` enables scraping these terms from the UNIPROT database. The data will be stored in a dictionary inside the `Analysis` class. *Technical note: this operation uses multi-threading for speed up.*

In [ ]:
analysis.GO_get_data()

Once the GO Terms of all the analysed proteins have been scraped, they can be interrogated with two different methods. The first searches for all the protein structures matching a specific GO code. Below, we will look for all lysines belonging to proteins associated with the GO term [0042803](https://www.ebi.ac.uk/QuickGO/term/GO:0042803) ("*dimerization activity*").

In [ ]:
analysis.GO_search_term(analysis.df, code ='0042803', name ='')

The second method extracts the GO Terms of a specific protein, defined by its UNIPROT code. Below, we retrieve all the GO termns of the protein P73832.

In [ ]:
GO_terms = analysis.GO_search_protein('P73832')
GO_terms #show data

We will now look for enrichment of GO terms within a specific pKa and SASA region. This enables us to determine whether specific GO terms are more prevalent in specific regions of the pKa/SASA space.

In [ ]:
pka_range = [1.0, 9.3]
sasa_range = [21.9, 100.0]
df = analysis.enrichment_analysis(pka_range, sasa_range)
df # show data

## 4. Interactive Data Visualization  <a class="anchor" id="plot"></a>

Once we have scraped all the GO Terms, we pass our `Analysis` object to a new class: `Viewer`. Let's start by initialising it. To do so, an aditional parameter is required: `outdir`. This informs viewer about where the protein structures associated with the data in `Analysis` are located (having protein structures available is a requirement).

In [ ]:
cp = Viewer(analysis, outdir=outdir)

Once initialised, the `Viewer` offers the `advanced_plot` method, usable in the following ways:

* without selecting any region, one can examine a GO Term shown in the dropdown lists and then click one of the red points (these are the average or the `south_east` measurement of the lysines that come from the proteins that have the GO Term). Finally, click a green point (an actual measurement of a lysine based on a particular PDB file) to see the 3D structure.
* one can also first select a region using the slides of pka and sasa. Note this operation will automatically initiate enrichment analysis. A bar chart will be produced based on the results of enrichment analysis. Finally, one can carry on the same process discussed above.
* If a region is selected, the export button can save the data (including GO Terms) within the region to a csv file and store it in the outdir defined by the user
* If a region is selected, the reset button can clear the region and the results of enrichment analysis
* If a 3D visualisation is displayed, the clear button can clear it.

The `cutoff` parameter is optional, and helps defining which GO terms entries should be ignored in bar chart plots (do not show any case appearing in the region of interest less or equal than this number of times, default value is 0).

In [ ]:
cp.advanced_plot(export_path='Regional_Data.csv', cutoff=0)

---